# module-base-class-custom — worked example 3: Reassigning an attribute moves it between registries

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-base-class-custom`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Because registration happens in `__setattr__`, reassigning a name with a different kind of value must move it between `_parameters` and `_modules` (popping the old slot). This keeps `parameters()` accurate when an attribute is overwritten.

## Worked solution

We show the registries stay consistent under reassignment.

1. **Assign a Parameter.** `self.x = Parameter(...)` lands in `_parameters['x']`.
2. **Reassign as a Module.** Setting `self.x = SubModule()` must remove `'x'` from `_parameters` and add it to `_modules`; otherwise `parameters()` would double-count or report a stale tensor.
3. **The pop.** Each branch of `__setattr__` pops the name from the other registry, guaranteeing a name lives in at most one place.
4. **Check counts.** After reassignment, the direct parameter count drops and the submodule's parameters now appear via recursion.

The demo assigns a Parameter, prints the param count, reassigns the same name to a submodule, and prints the updated count.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    def __init__(self):
        object.__setattr__(self, '_parameters', {})
        object.__setattr__(self, '_modules', {})
    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._parameters[name] = value
            self._modules.pop(name, None)
        elif isinstance(value, Module):
            self._modules[name] = value
            self._parameters.pop(name, None)
        else:
            self._parameters.pop(name, None)
            self._modules.pop(name, None)
        object.__setattr__(self, name, value)
    def parameters(self):
        for p in self._parameters.values():
            yield p
        for m in self._modules.values():
            yield from m.parameters()

class Sub(Module):
    def __init__(self):
        super().__init__()
        self.w = Parameter(np.ones(4))

m = Module()
m.x = Parameter(np.ones(3))
print('after param assign — direct params:', len(m._parameters), '| total:', len(list(m.parameters())))
m.x = Sub()
print('after module reassign — direct params:', len(m._parameters), '| modules:', len(m._modules), '| total:', len(list(m.parameters())))